In [13]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

/home/joaquin/Documents/GitHub/skforecast
0.22.0


## tabicl

In [14]:
# !pip install tabicl

In [17]:
# Libraries
# ==============================================================================
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from skforecast.datasets import load_demo_dataset
from skforecast.preprocessing import RollingFeatures
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import TimeSeriesFold
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import grid_search_forecaster
from skforecast.plot import set_dark_theme

# Download data
# ==============================================================================
data = load_demo_dataset()
data.head(5)

# Data partition train-test
# ==============================================================================
end_train = '2005-06-01 23:59:00'
print(
    f"Train dates : {data.index.min()} --- {data.loc[:end_train].index.max()}  " 
    f"(n={len(data.loc[:end_train])})"
)
print(
    f"Test dates  : {data.loc[end_train:].index.min()} --- {data.index.max()}  "
    f"(n={len(data.loc[end_train:])})"
)

╭────────────────────────────────────── h2o ───────────────────────────────────────╮
│ Description:                                                                     │
│ Monthly expenditure ($AUD) on corticosteroid drugs that the Australian health    │
│ system had between 1991 and 2008.                                                │
│                                                                                  │
│ Source:                                                                          │
│ Hyndman R (2023). fpp3: Data for Forecasting: Principles and Practice(3rd        │
│ Edition). http://pkg.robjhyndman.com/fpp3package/,https://github.com/robjhyndman │
│ /fpp3package, http://OTexts.com/fpp3.                                            │
│                                                                                  │
│ URL:                                                                             │
│ https://raw.githubusercontent.com/skforecast/skforecast-                         │
│ datasets/main/data/h2o.csv                                                       │
│                                                                                  │
│ Shape: 204 rows                                                                  │
╰──────────────────────────────────────────────────────────────────────────────────╯

Train dates : 1991-07-01 00:00:00 --- 2005-06-01 00:00:00  (n=168)
Test dates  : 2005-07-01 00:00:00 --- 2008-06-01 00:00:00  (n=36)


In [18]:
forecaster = ForecasterRecursive(
                 estimator       = TabICLRegressor(),
                 lags            = 15,
             )

forecaster.fit(y=data.loc[:end_train])
forecaster

=================== 
ForecasterRecursive 
=================== 
Estimator: TabICLRegressor 
Lags: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15] 
Window features: None 
Window size: 15 
Series name: y 
Exogenous included: False 
Exogenous names: None 
Categorical features: auto 
Transformer for y: None 
Transformer for exog: None 
Weight function included: False 
Differentiation order: None 
Training range: [Timestamp('1991-07-01 00:00:00'), Timestamp('2005-06-01 00:00:00')] 
Training index type: DatetimeIndex 
Training index frequency: <MonthBegin> 
Estimator parameters: 
    {'allow_auto_download': True, 'batch_size': 8, 'checkpoint_version': 'tabicl-
    regressor-v2-20260212.ckpt', 'device': None, 'disk_offload_dir': None,
    'feat_shuffle_method': 'latin', 'inference_config': None, 'kv_cache': False,
    'model_path': None, 'n_estimators': 8, 'n_jobs': None, 'norm_methods': None,
    'offload_mode': 'auto', 'outlier_threshold': 4.0, 'random_state': 42,
    'use_amp': 'auto', 'use_fa3': 'auto', 'verbose': False} 
fit_kwargs: {} 
Creation date: 2026-03-31 23:51:38 
Last fit date: 2026-03-31 23:51:39 
Skforecast version: 0.22.0 
Python version: 3.13.11 
Forecaster id: None

In [8]:
# Predict
# ==============================================================================
predictions = forecaster.predict(steps=len(data.loc[end_train:]))
predictions.head(3)

2005-07-01    0.914454
2005-08-01    0.974993
2005-09-01    1.075126
Freq: MS, Name: pred, dtype: float64

In [19]:
# Backtesting
# ==============================================================================
cv = TimeSeriesFold(
         steps              = 10,
         initial_train_size = len(data.loc[:end_train]),
         refit              = True,
         fixed_train_size   = False
     )

metric, predictions_backtest = backtesting_forecaster(
                                   forecaster    = forecaster,
                                   y             = data,
                                   cv            = cv,
                                   metric        = 'mean_squared_error',
                                   verbose       = True,
                               )

display(metric)
predictions_backtest

Information of folds
--------------------
Number of observations used for initial training: 168
Number of observations used for backtesting: 36
    Number of folds: 4
    Number skipped folds: 0 
    Number of steps per fold: 10
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 6 observations.

Fold: 0
    Training:   1991-07-01 00:00:00 -- 2005-06-01 00:00:00  (n=168)
    Validation: 2005-07-01 00:00:00 -- 2006-04-01 00:00:00  (n=10)
Fold: 1
    Training:   1991-07-01 00:00:00 -- 2006-04-01 00:00:00  (n=178)
    Validation: 2006-05-01 00:00:00 -- 2007-02-01 00:00:00  (n=10)
Fold: 2
    Training:   1991-07-01 00:00:00 -- 2007-02-01 00:00:00  (n=188)
    Validation: 2007-03-01 00:00:00 -- 2007-12-01 00:00:00  (n=10)
Fold: 3
    Training:   1991-07-01 00:00:00 -- 2007-12-01 00:00:00  (n=198)
    Validation: 2008-01-01 00:00:00 -- 2008-06-01 00:00:00  (n=6)



  0%|          | 0/4 [00:00<?, ?it/s]

,mean_squared_error
0,0.004761


,fold,pred
2005-07-01,0,0.914454
2005-08-01,0,0.974992
2005-09-01,0,1.075125
2005-10-01,0,1.118152
2005-11-01,0,1.152766
2005-12-01,0,1.190976
2006-01-01,0,1.130322
2006-02-01,0,0.598897
2006-03-01,0,0.693380
2006-04-01,0,0.679212


## tabpfn

In [10]:
# pip install tabpfn
from tabpfn import TabPFNRegressor

In [11]:
forecaster = ForecasterRecursive(
                 estimator       = TabPFNRegressor(),
                 lags            = 15,
             )

forecaster.fit(y=data.loc[:end_train])
forecaster

RuntimeError: Failed to download TabPFN ModelVersion.V2_6 model 'tabpfn-v2.6-regressor-v2.6_default.ckpt'.

Details and instructions:
HuggingFace authentication error downloading from 'Prior-Labs/tabpfn_2_6'.
This model is gated and requires you to accept its terms.

Please follow these steps:
1. Visit https://huggingface.co/Prior-Labs/tabpfn_2_6 in your browser and accept the terms of use.
2. Log in to your Hugging Face account via the command line by running:
   hf auth login
   (Alternatively, you can set the HF_TOKEN environment variable   with a read token.)

For detailed instructions, see https://docs.priorlabs.ai/how-to-access-gated-models

For commercial usage, we provide alternative download options for TabPFN ModelVersion.V2_6; please reach out to us at sales@priorlabs.ai.